# 06 — Exceptions

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- lire et comprendre une traceback Python ;
- attraper une exception avec `try` / `except` ;
- utiliser `else` et `finally` ;
- lever une exception avec `raise` ;
- définir votre propre classe d'exception.

## Prérequis

- toutes les notions précédentes ;
- fonctions typées, closures.

Pas encore vus :

- débogueur `pdb` (notebook suivant) ;
- comprehensions et générateurs.

## Plan

1. Qu'est-ce qu'une exception ?
2. Lire une traceback
3. `try` / `except`
4. Attraper plusieurs types
5. `else` et `finally`
6. `raise`
7. Chaîner : `raise ... from ...`
8. Exceptions personnalisées
9. Règles de conception
10. Synthèse
11. Exercices

---


## 1. Qu'est-ce qu'une exception ?

Une **exception** est un événement qui interrompt le flux normal d'exécution. En Python, c'est un **objet** qui hérite de `Exception`.

In [ ]:
1 / 0

In [ ]:
int('abc')

In [ ]:
d = {'a': 1}
d['z']

---


## 2. Lire une traceback

Une traceback se lit **de haut en bas** : chaque ligne est une étape d'appel. La **dernière** ligne donne le type d'exception et le message — c'est souvent la plus utile.

In [ ]:
def niveau1() -> int:
    return niveau2()

def niveau2() -> int:
    return int('abc')

niveau1()

---


## 3. `try` / `except`

Le bloc `try` contient le code à surveiller. Si une exception est levée, l'exécution bascule dans le premier `except` qui correspond.

In [ ]:
def division_sure(a: float, b: float) -> float | None:
    try:
        return a / b
    except ZeroDivisionError:
        return None

print(division_sure(10, 2))
print(division_sure(10, 0))

### Récupérer l'exception avec `as`

In [ ]:
def parser_int(texte: str) -> int | None:
    try:
        return int(texte)
    except ValueError as err:
        print('erreur de parsing :', err)
        return None

parser_int('abc')

---


## 4. Attraper plusieurs types

In [ ]:
def operation(a: int, b: int, op: str) -> int | None:
    try:
        if op == '+':
            return a + b
        if op == '/':
            return a // b
        raise ValueError(f'opération inconnue : {op}')
    except (ZeroDivisionError, ValueError) as err:
        print('erreur :', err)
        return None

operation(10, 0, '/')
operation(10, 2, '?')

### Plusieurs `except` successifs

In [ ]:
def lire_entier(texte: str) -> int | None:
    try:
        return int(texte)
    except ValueError:
        print('pas un entier valide')
        return None
    except TypeError:
        print('type inattendu')
        return None

---


## 5. `else` et `finally`

- `else` : exécuté **uniquement** si le `try` n'a levé aucune exception.
- `finally` : exécuté **toujours** (succès, échec, exception non attrapée).

In [ ]:
def diviser(a: float, b: float) -> None:
    try:
        resultat = a / b
    except ZeroDivisionError:
        print('division par zéro')
    else:
        print('résultat :', resultat)
    finally:
        print('nettoyage')

diviser(10, 2)
print('---')
diviser(10, 0)

### Pourquoi `else` ?

Pour restreindre le `try` au strict minimum — éviter d'y mettre du code qui ne lève pas l'exception attendue.

---


## 6. `raise` — lever soi-même une exception

On peut lever une exception explicitement avec `raise`. C'est utile quand les arguments sont invalides.

In [ ]:
def racine_carree(x: float) -> float:
    if x < 0:
        raise ValueError(f'x doit être >= 0, reçu {x}')
    return x ** 0.5

print(racine_carree(9))
racine_carree(-1)

### Re-lever l'exception en cours

Un `raise` sans argument dans un `except` re-lève l'exception actuelle, telle quelle.

In [ ]:
def traiter() -> None:
    try:
        int('abc')
    except ValueError:
        print('je logge avant de laisser remonter')
        raise

try:
    traiter()
except ValueError as err:
    print('remontée :', err)

---


## 7. Chaîner : `raise ... from ...`

Quand on convertit une exception en une autre, `from` préserve la cause d'origine dans la traceback.

In [ ]:
class ConfigError(Exception):
    pass

def charger_port(texte: str) -> int:
    try:
        return int(texte)
    except ValueError as err:
        raise ConfigError(f'port invalide : {texte!r}') from err

try:
    charger_port('abc')
except ConfigError as err:
    print('erreur config :', err)
    print('cause :', err.__cause__)

---


## 8. Exceptions personnalisées

On définit une exception métier en héritant de `Exception`. Un `pass` suffit au corps — le nom suffit à documenter.

In [ ]:
class SoldeInsuffisant(Exception):
    """Levée quand un retrait dépasse le solde disponible."""
    pass

In [ ]:
def retirer(solde: float, montant: float) -> float:
    if montant > solde:
        raise SoldeInsuffisant(f'demande {montant}, solde {solde}')
    return solde - montant

try:
    retirer(100.0, 250.0)
except SoldeInsuffisant as err:
    print('refusé :', err)

### Enrichir avec des attributs

In [ ]:
class SoldeInsuffisant(Exception):
    def __init__(self, solde: float, montant: float) -> None:
        super().__init__(f'demande {montant}, solde {solde}')
        self.solde = solde
        self.montant = montant

try:
    raise SoldeInsuffisant(100.0, 250.0)
except SoldeInsuffisant as err:
    print('manque :', err.montant - err.solde)

---


## 9. Règles de conception

1. **Ne pas attraper `Exception` aveuglément** — c'est cacher les bugs.
2. Attraper **le type le plus précis possible**.
3. Ne pas **ignorer** une exception avec un `except: pass` muet.
4. Préférer `raise NouvelleErreur('...') from cause` pour garder le contexte.
5. Une exception est **exceptionnelle** : pas pour gérer des flots normaux.

---


## 10. Synthèse

| Élément | Rôle |
|---|---|
| `try:` | Bloc surveillé |
| `except E as err:` | Attrape `E` et lie à `err` |
| `except (A, B):` | Attrape `A` ou `B` |
| `else:` | Si aucun `except` ne s'est déclenché |
| `finally:` | Toujours exécuté |
| `raise E('msg')` | Lève une exception |
| `raise E(...) from cause` | Chaînage |
| `class E(Exception):` | Exception personnalisée |

---


## 11. Exercices

### Exercice 1 — Parsage sûr *(facile)*

Écrire `parser_int_sur(texte: str) -> int | None` qui renvoie l'entier parsé, ou `None` si la conversion échoue.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="06_Exceptions", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
def parser_int_sur(texte: str) -> int | None:
    """Renvoie int(texte) ou None en cas d'échec."""
    try:
        return int(texte)
    except ValueError:
        return None

print(parser_int_sur('42'))
print(parser_int_sur('abc'))
```

</details>

### Exercice 2 — Division protégée *(facile)*

Écrire `division(a: float, b: float) -> float` qui lève `ValueError('division par zéro')` si `b == 0`, sinon renvoie `a / b`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="06_Exceptions", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
def division(a: float, b: float) -> float:
    """Division protégée."""
    if b == 0:
        raise ValueError('division par zéro')
    return a / b

try:
    division(10, 0)
except ValueError as err:
    print(err)
```

</details>

### Exercice 3 — Accès clé tolérant *(facile)*

Écrire `lire(d: dict[str, int], cle: str) -> int` qui renvoie la valeur, ou affiche `'clé absente'` et renvoie `0` si la clé est manquante. Utiliser `try`/`except KeyError`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="06_Exceptions", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
def lire(d: dict[str, int], cle: str) -> int:
    try:
        return d[cle]
    except KeyError:
        print('clé absente')
        return 0

print(lire({'a': 1}, 'a'))
print(lire({'a': 1}, 'z'))
```

</details>

### Exercice 4 — Moyenne robuste *(moyen)*

Écrire `moyenne(valeurs: list[float]) -> float` qui lève une exception `ValueError('liste vide')` si la liste est vide, sinon renvoie la moyenne.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="06_Exceptions", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
def moyenne(valeurs: list[float]) -> float:
    """Moyenne d'une liste non vide."""
    if len(valeurs) == 0:
        raise ValueError('liste vide')
    return sum(valeurs) / len(valeurs)

print(moyenne([10.0, 20.0, 30.0]))
try:
    moyenne([])
except ValueError as err:
    print('attrapé :', err)
```

</details>

### Exercice 5 — Exception personnalisée *(moyen)*

Définir `class AgeInvalide(Exception): pass`. Écrire `verifier_age(age: int) -> None` qui lève `AgeInvalide` si `age` n'est pas dans `[0, 130]`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="06_Exceptions", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
class AgeInvalide(Exception):
    """Âge hors intervalle raisonnable."""
    pass

def verifier_age(age: int) -> None:
    if age < 0 or age > 130:
        raise AgeInvalide(f'âge invalide : {age}')

try:
    verifier_age(200)
except AgeInvalide as err:
    print(err)
```

</details>

### Exercice 6 — Retry *(moyen)*

Écrire `boucle_saisie() -> int` qui demande à l'utilisateur un entier, jusqu'à ce qu'il fournisse une réponse valide. À chaque échec, afficher `'réessayez'`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="06_Exceptions", exercice=6)


<details>
<summary>📖 Voir la correction</summary>

```python
def boucle_saisie() -> int:
    while True:
        texte = input('Entier : ')
        try:
            return int(texte)
        except ValueError:
            print('réessayez')

# Appel commenté pour éviter un input dans les tests
# print(boucle_saisie())
```

</details>

### Exercice 7 — Compte bancaire *(difficile)*

Définir une exception `SoldeInsuffisant(Exception)`. Écrire `retirer(solde: float, montant: float) -> float` qui :
- lève `ValueError` si `montant < 0` ;
- lève `SoldeInsuffisant` si `montant > solde` ;
- renvoie le nouveau solde sinon.

Tester les trois cas dans un bloc `try/except`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="06_Exceptions", exercice=7)


<details>
<summary>📖 Voir la correction</summary>

```python
class SoldeInsuffisant(Exception):
    pass

def retirer(solde: float, montant: float) -> float:
    if montant < 0:
        raise ValueError(f'montant négatif : {montant}')
    if montant > solde:
        raise SoldeInsuffisant(f'{montant} > {solde}')
    return solde - montant

for essai in [(100.0, 30.0), (100.0, -5.0), (100.0, 500.0)]:
    try:
        solde, m = essai
        print('nouveau solde :', retirer(solde, m))
    except (ValueError, SoldeInsuffisant) as err:
        print(type(err).__name__, ':', err)
```

</details>

### Exercice 8 — Chaînage `raise from` *(difficile)*

Définir `class ConfigError(Exception): pass`. Écrire `charger_port(texte: str) -> int` qui tente `int(texte)` et, en cas d'échec, lève `ConfigError('port invalide : ...')` `from` la ValueError. Dans le test, afficher `err.__cause__`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="06_Exceptions", exercice=8)


<details>
<summary>📖 Voir la correction</summary>

```python
class ConfigError(Exception):
    pass

def charger_port(texte: str) -> int:
    try:
        return int(texte)
    except ValueError as err:
        raise ConfigError(f'port invalide : {texte!r}') from err

try:
    charger_port('abc')
except ConfigError as err:
    print('erreur :', err)
    print('cause :', repr(err.__cause__))
```

</details>

---


## Ressources externes

- [Exceptions — tutoriel](https://docs.python.org/3/tutorial/errors.html)
- [Hiérarchie des exceptions builtin](https://docs.python.org/3/library/exceptions.html#exception-hierarchy)

---

## Mini-exemples supplémentaires

### `assert` — vérification de contrat

In [ ]:
def racine(x: float) -> float:
    assert x >= 0, 'x doit être positif'
    return x ** 0.5

print(racine(4))

`assert` lève `AssertionError` si la condition est fausse. À utiliser pour des invariants internes, **pas** pour valider des entrées utilisateur (un utilisateur peut lancer Python avec `-O` qui désactive les assertions).

In [ ]:
try:
    racine(-1)
except AssertionError as err:
    print('attrapé :', err)

### Capturer et ignorer — `except: pass` est un anti-pattern

In [ ]:
def lire_silence(chemin: str) -> str:
    try:
        with open(chemin, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return ''  # silence explicite et justifié

On **loggue** ou on **retourne une valeur par défaut**, mais on n'ignore jamais une exception sans raison.

### Hiérarchie des exceptions builtin

In [ ]:
print(ValueError.__mro__)

`ValueError` hérite de `Exception` qui hérite de `BaseException`. **Ne jamais** attraper `BaseException` directement — vous attraperiez aussi `KeyboardInterrupt` et `SystemExit`.

In [ ]:
# Liste (partielle) des exceptions courantes
for e in (ValueError, TypeError, KeyError, IndexError,
          FileNotFoundError, ZeroDivisionError, AttributeError):
    print(e.__name__)

### `ExceptionGroup` (Python 3.11+)

In [ ]:
# ExceptionGroup permet de lever plusieurs exceptions en une fois
# try:
#     raise ExceptionGroup('plusieurs erreurs', [ValueError('a'), KeyError('b')])
# except* ValueError as eg:
#     print('valeur:', eg)
print('voir doc 3.11+')